# Isaac Sim on Google Colab

Unofficial instructions for running headless [Isaac Sim](https://docs.isaacsim.omniverse.nvidia.com/latest/index.html) and [Isaac Lab](https://isaac-sim.github.io/IsaacLab/main/index.html) on Google Colab.

This setup is for demo purposes only, using various hacks to run Isaac Sim on Colab. Serious development is not recommended, as Colab is not officially supported.

**First save a copy in your own Google Drive: `File > Save a copy in Drive`.**

## 1. Install Isaac Sim and Isaac Lab

**Make sure to first change to GPU runtime by `Runtime > Change runtime type` to `T4 GPU`.**

> Please note that error messages indicating Isaac Sim was terminated due to segmentation faults or Python fatal errors can be ignored.

In [ ]:
# Install Isaac Sim 5.1 and Isaac Lab 2.3.2. This process takes about 10 mins to complete.
!wget -O install-isaac-sim-5.1.sh https://raw.githubusercontent.com/kaweees/isaac-sim-colab/main/scripts/install-isaac-sim-5.1.sh
!time bash install-isaac-sim-5.1.sh
!wget -O install-isaac-lab-2.3.2.sh https://raw.githubusercontent.com/kaweees/isaac-sim-colab/main/scripts/install-isaac-lab-2.3.2.sh
!time bash install-isaac-lab-2.3.2.sh

In [ ]:
# Install common packages
!pip install seaborn tbparse

## 2. Train an Agent in the Unitree Go2 Environment

Begin the training below, and in the meantime, review the code here: [Unitree Go2 Environment Code](https://github.com/isaac-sim/IsaacLab/blob/main/source/isaaclab_tasks/isaaclab_tasks/direct/locomotion/locomotion_env.py).

In [ ]:
# Set environment variables
# Ref: https://docs.isaacsim.omniverse.nvidia.com/latest/installation/install_python.html#running-isaac-sim
import os
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"

# Train agent in Unitree Go2 env
# Ref: https://isaac-sim.github.io/IsaacLab/main/source/overview/reinforcement-learning/rl_existing_scripts.html
!python -m isaaclab.train --task=Isaac-Velocity-Rough-Unitree-Go2-v0 --num_envs 4096 --headless --video --video_length 100 --enable_cameras --logger=tensorboard --output_name=model --export_onnx --export_jit

In [ ]:
# Set environment variables
# Ref: https://docs.isaacsim.omniverse.nvidia.com/latest/installation/install_python.html#running-isaac-sim
import os
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"

# Record agent in Unitree Go2 env
# Ref: https://isaac-sim.github.io/IsaacLab/main/source/overview/reinforcement-learning/rl_existing_scripts.html
!python -m isaaclab.play --task=Isaac-Velocity-Rough-Unitree-Go2-v0 --headless --video --video_length 200

# Need to click the `Stop` button twice after recording finished (wait for the video file), not sure why the script won't end automatically by itself.

In [ ]:
import glob
from IPython.display import Video

pattern = "logs/rsl_rl/Isaac-Velocity-Rough-Unitree-Go2-v0/*/videos/play/rl-video-step-0.mp4"
files = glob.glob(pattern)

# Show Unitree Go2 recording
Video(files[0], embed=True)

**Please refresh this page by pressing F5, and then execute the following cell without running the first one.**

If execution later appears stuck (showing "running" but not progressing), refreshing the page with F5 usually resolves the issue.

In [ ]:
import sys
print("User Current Version:-", sys.version)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

from tbparse import SummaryReader
reader = SummaryReader("logs/rsl_rl/Isaac-Velocity-Rough-Unitree-Go2-v0")
df = reader.scalars

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.lineplot(ax=axes[0], data=df[df["tag"] == "rewards/step"], x='step', y='value', hue='tag')
axes[0].set_title('Reward vs. Steps')
sns.lineplot(ax=axes[1], data=df[df["tag"] == "rewards/iter"], x='step', y='value', hue='tag')
axes[1].set_title('Reward vs. Iters')
sns.lineplot(ax=axes[2], data=df[df["tag"] == "rewards/time"], x='step', y='value', hue='tag')
axes[2].set_title('Reward vs. Time')

## 3. Train an Agent in the Ant Environment

Begin the training below, and in the meantime, review the code here: [Ant Environment Code](https://github.com/isaac-sim/IsaacLab/blob/main/source/isaaclab_tasks/isaaclab_tasks/direct/ant/ant_env.py) and [Locomotion Environment Code](https://github.com/isaac-sim/IsaacLab/blob/main/source/isaaclab_tasks/isaaclab_tasks/direct/locomotion/locomotion_env.py).

In [ ]:
# Set environment variables
# Ref: https://docs.isaacsim.omniverse.nvidia.com/latest/installation/install_python.html#running-isaac-sim
import os
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"

# Train agent in Ant env
# Ref: https://isaac-sim.github.io/IsaacLab/main/source/overview/reinforcement-learning/rl_existing_scripts.html
!python -m isaaclab.train --task Isaac-Ant-v0 --headless

In [ ]:
# Set environment variables
# Ref: https://docs.isaacsim.omniverse.nvidia.com/latest/installation/install_python.html#running-isaac-sim
import os
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"

# Record agent in Ant env
# Ref: https://isaac-sim.github.io/IsaacLab/main/source/overview/reinforcement-learning/rl_existing_scripts.html
!python -m isaaclab.play --task Isaac-Ant-v0 --headless --video --video_length 400

# Need to click the `Stop` button twice after recording finished (wait for the video file), not sure why the script won't end automatically by itself.

In [ ]:
import glob
from IPython.display import Video

pattern = "logs/rl_games/ant/*/videos/play/rl-video-step-0.mp4"
files = glob.glob(pattern)

# Show Ant recording
Video(files[0], embed=True)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

from tbparse import SummaryReader
reader = SummaryReader("logs/rl_games/ant")
df = reader.scalars

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.lineplot(ax=axes[0], data=df[df["tag"] == "rewards/step"], x='step', y='value', hue='tag')
axes[0].set_title('Reward vs. Steps')
sns.lineplot(ax=axes[1], data=df[df["tag"] == "rewards/iter"], x='step', y='value', hue='tag')
axes[1].set_title('Reward vs. Iters')
sns.lineplot(ax=axes[2], data=df[df["tag"] == "rewards/time"], x='step', y='value', hue='tag')
axes[2].set_title('Reward vs. Time')

## 4. Pretrained Checkpoints

In [ ]:
# Set environment variables
# Ref: https://docs.isaacsim.omniverse.nvidia.com/latest/installation/install_python.html#running-isaac-sim
import os
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"

# Record pretrained agent
# Ref: https://isaac-sim.github.io/IsaacLab/main/source/overview/reinforcement-learning/rl_existing_scripts.html
!python -m isaaclab.play --task Isaac-Open-Drawer-Franka-v0 --num_envs 32 --headless --video --video_length 600 --use_pretrained_checkpoint

# Need to click the `Stop` button twice after recording finished (wait for the video file), not sure why the script won't end automatically by itself.

In [ ]:
import glob
from IPython.display import Video

pattern = ".pretrained_checkpoints/rl_games/videos/play/rl-video-step-0.mp4"
files = glob.glob(pattern)

# Show pretrained agent recording
Video(files[0], embed=True)

In [ ]:
# Set environment variables
# Ref: https://docs.isaacsim.omniverse.nvidia.com/latest/installation/install_python.html#running-isaac-sim
import os
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"

# Record pretrained agent
# Ref: https://isaac-sim.github.io/IsaacLab/main/source/overview/reinforcement-learning/rl_existing_scripts.html
!python -m isaaclab.play --task Isaac-Velocity-Rough-Unitree-Go2-v0 --num_envs 1024 --headless --video --video_length 600 --use_pretrained_checkpoint

# Need to click the `Stop` button twice after recording finished (wait for the video file), not sure why the script won't end automatically by itself.

In [ ]:
import glob
from IPython.display import Video

pattern = ".pretrained_checkpoints/rsl_rl/Isaac-Velocity-Rough-Unitree-Go2-v0/videos/play/rl-video-step-0.mp4"
files = glob.glob(pattern)

# Show pretrained agent recording
Video(files[0], embed=True)

In [ ]:
# Set environment variables
# Ref: https://docs.isaacsim.omniverse.nvidia.com/latest/installation/install_python.html#running-isaac-sim
import os
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"

# Record pretrained agent
# Ref: https://isaac-sim.github.io/IsaacLab/main/source/overview/reinforcement-learning/rl_existing_scripts.html
!python -m isaaclab.play --task Isaac-Velocity-Rough-H1-v0 --num_envs 1024 --headless --video --video_length 600 --use_pretrained_checkpoint

# Need to click the `Stop` button twice after recording finished (wait for the video file), not sure why the script won't end automatically by itself.

In [ ]:
import glob
from IPython.display import Video

pattern = ".pretrained_checkpoints/rsl_rl/Isaac-Velocity-Rough-H1-v0/videos/play/rl-video-step-0.mp4"
files = glob.glob(pattern)

# Show pretrained agent recording
Video(files[0], embed=True)

## 5. More Environments

See [Isaac Lab Available Environments](https://isaac-sim.github.io/IsaacLab/main/source/overview/environments.html) for the full list of environments.

In [ ]:
# Set environment variables
# Ref: https://docs.isaacsim.omniverse.nvidia.com/latest/installation/install_python.html#running-isaac-sim
import os
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"

!python -m isaaclab.list_envs